# Imports and Data Read

In [256]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os

In [257]:
df = pd.read_csv('/Users/ziadsamer/Documents/who-will-viral/data/youtube/extracted.csv')

# Categorical Features

In [258]:
df.select_dtypes(include=['object']).columns

/var/folders/p6/82pzwvfj08b5bktj5wbwqwh00000gn/T/ipykernel_19810/549554427.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=['object']).columns


Index(['video_id', 'title', 'publishedAt', 'channelId', 'channelTitle', 'tags',
       'description', 'defaultLanguage', 'duration', 'dimension', 'definition',
       'projection', 'playability_status', 'tags_joined'],
      dtype='str')

In [259]:
df['lang_base'] = df['defaultLanguage'].str.split('-').str[0]
df['lang_region'] = df['defaultLanguage'].str.split('-').str[1]

/var/folders/p6/82pzwvfj08b5bktj5wbwqwh00000gn/T/ipykernel_19810/3004818695.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['lang_base'] = df['defaultLanguage'].str.split('-').str[0]
/var/folders/p6/82pzwvfj08b5bktj5wbwqwh00000gn/T/ipykernel_19810/3004818695.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['lang_region'] = df['defaultLanguage'].str.split('-').str[1]


In [260]:
df['lang_base'].value_counts()

lang_base
en     146451
es       5032
hi       4014
fr       3017
id       2627
        ...  
vro         1
mo          1
bho         1
bi          1
mni         1
Name: count, Length: 87, dtype: int64

In [261]:
df['lang_region'] = df['lang_region'].fillna('NO')

In [262]:
df['lang_region'].value_counts() # rename 419 another thing

lang_region
NO      162993
US        6278
IN        4786
GB        3816
PT        1155
419        519
BR         290
CA         149
ES         110
Hant       109
FR          87
MX          86
Latn        55
AU          43
DE          40
TW          31
AT          15
NL          14
CN          13
IE          12
BE          10
HK           8
Hans         4
CH           3
SG           1
Name: count, dtype: int64

In [263]:
threshold = 0.002

freq = df['lang_base'].value_counts(normalize=True)
rare = freq[freq < threshold].index

df['lang_base'] = df['lang_base'].replace(rare, 'other')

In [264]:
threshold = 0.005

freq = df['lang_region'].value_counts(normalize=True)
rare = freq[freq < threshold].index

df['lang_region'] = df['lang_region'].replace(rare, 'other')

In [265]:
df['lang_region'].value_counts() # Then frequency encoding

lang_region
NO       162993
US         6278
IN         4786
GB         3816
other      1599
PT         1155
Name: count, dtype: int64

In [266]:
df['lang_base'].value_counts() # Then frequency encoding

lang_base
en       146451
es         5032
hi         4014
other      3158
fr         3017
id         2627
vi         2146
ko         1982
pt         1613
de         1613
ja         1463
tr         1136
ru         1098
bn         1080
it          854
th          736
ar          682
ml          679
pl          653
ta          593
Name: count, dtype: int64

In [267]:
df['dimension'].value_counts() # I think drop it

dimension
2d    180610
3d        17
Name: count, dtype: int64

In [268]:
df['definition'].value_counts() # severe imbalance I think drop it

definition
hd    179572
sd      1055
Name: count, dtype: int64

In [269]:
df['playability_status'].value_counts() # I think drop it

playability_status
UNPLAYABLE                175008
OK                          5246
LOGIN_REQUIRED               345
CONTENT_CHECK_REQUIRED        22
ERROR                          6
Name: count, dtype: int64

In [270]:
df['supports_miniplayer'].value_counts() # I think drop it

supports_miniplayer
True     180252
False       375
Name: count, dtype: int64

In [271]:
df['projection'].value_counts() # I think drop it

projection
rectangular    180597
360                30
Name: count, dtype: int64

In [272]:
COLS_TO_DROP = [
    'videoid', 'title', 'publishedAt', 'channelId', 
    'tags',  'likes', 'comment_count', 'description',
    'duration', 'tags_joined', 'defaultLanguage', 'supports_miniplayer',
    'playability_status', 'definition', 'dimension', 'projection',
    'channelTitle', "has_paid_promotion", "card_count"
]

## Numerical columns

In [273]:
# categoryId,
# chapter_count

In [274]:
df.select_dtypes(include=['int64', 'float64']).columns

Index(['categoryId', 'view_count', 'likes', 'comment_count', 'is_trending',
       'chapter_count', 'card_count', 'like_to_view_ratio',
       'comment_to_view_ratio', 'tag_count',
       ...
       'title_emb_374', 'title_emb_375', 'title_emb_376', 'title_emb_377',
       'title_emb_378', 'title_emb_379', 'title_emb_380', 'title_emb_381',
       'title_emb_382', 'title_emb_383'],
      dtype='str', length=1170)

In [275]:
cap = 20

df['chapter_count_capped'] = df['chapter_count'].clip(upper=cap)

/var/folders/p6/82pzwvfj08b5bktj5wbwqwh00000gn/T/ipykernel_19810/3982712297.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['chapter_count_capped'] = df['chapter_count'].clip(upper=cap)


In [276]:
q1 = df['chapter_count'].quantile(0.25)
q3 = df['chapter_count'].quantile(0.75)
iqr = q3 - q1

df_2 = df[(df['chapter_count'] >= q1 - 1.5 * iqr) & (df['chapter_count'] <= q3 + 1.5 * iqr)]

df_2['chapter_count'].value_counts()

chapter_count
0    161836
Name: count, dtype: int64

In [277]:
df['has_chapter'] = df['chapter_count'] > 0

/var/folders/p6/82pzwvfj08b5bktj5wbwqwh00000gn/T/ipykernel_19810/1145261948.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_chapter'] = df['chapter_count'] > 0


In [278]:
df['card_count'].value_counts()

card_count
0    175538
1      5089
Name: count, dtype: int64

In [279]:
df['categoryId'].value_counts()

categoryId
22    38331
24    38320
27    17353
20    15034
10    13009
26    12550
28    10181
17    10129
23     6195
1      6005
25     5411
19     4149
2      2578
15     1119
29      251
30       12
Name: count, dtype: int64

In [280]:
min_count = 500

counts = df['categoryId'].value_counts()
rare = counts[counts < min_count].index

df['categoryId'] = df['categoryId'].replace(rare, 29)

In [281]:
df['comments_disabled'].value_counts()

comments_disabled
False    171684
True       8943
Name: count, dtype: int64

In [282]:
df['has_cards'] = (df['card_count'] > 0).astype(int)

/var/folders/p6/82pzwvfj08b5bktj5wbwqwh00000gn/T/ipykernel_19810/624939411.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_cards'] = (df['card_count'] > 0).astype(int)


In [283]:
df['is_verified'].value_counts()

is_verified
True     111453
False     69174
Name: count, dtype: int64

# Drop 

In [284]:
COLS_TO_DROP = [
    'video_id', 'title', 'publishedAt', 'channelId', 
    'tags',  'likes', 'comment_count', 'description',
    'duration', 'tags_joined', 'defaultLanguage', 'supports_miniplayer',
    'playability_status', 'definition', 'dimension', 'projection',
    'channelTitle', "has_paid_promotion", "card_count"
]

In [285]:
df.drop(COLS_TO_DROP, axis=1, inplace=True)

# Split Data

In [286]:
from sklearn.model_selection import train_test_split

In [287]:
def _split_data(df):
    df = df.reset_index(drop=True)
    
    feature_cols = [c for c in df.columns if c != "is_trending"]
    
    X = df[feature_cols].values
    y = df["is_trending"].values

    X_temp, X_test, y_temp, y_test = train_test_split(  # correct order
        X, y, test_size=0.2, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=42
    )

    def to_df(X_arr, y_arr):
        out = pd.DataFrame(X_arr, columns=feature_cols)
        out["is_trending"] = y_arr
        return out

    return to_df(X_train, y_train), to_df(X_val, y_val), to_df(X_test, y_test)


In [288]:
def _split_data(df):
    df = df.reset_index(drop=True)

    temp, test = train_test_split(df, test_size=0.2, random_state=42)
    train, val = train_test_split(temp, test_size=0.25, random_state=42)

    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )

In [289]:
df_train, df_val, df_test = _split_data(df)

# Feature Encoding

In [290]:
freq_base = df_train['lang_base'].value_counts(normalize=True)
freq_region = df_train['lang_region'].value_counts(normalize=True)

In [291]:
df_train['lang_base'] = df_train['lang_base'].map(freq_base)
df_val['lang_base'] = df_val['lang_base'].map(freq_base)
df_test['lang_base'] = df_test['lang_base'].map(freq_base)

In [292]:
df_train['lang_region'] = df_train['lang_region'].map(freq_region)
df_val['lang_region'] = df_val['lang_region'].map(freq_region)
df_test['lang_region'] = df_test['lang_region'].map(freq_region)

# Feature Transformation

In [293]:
from sklearn.decomposition import PCA

In [294]:
TO_TRANSFORM = [col for col in df_train.columns if 'emb' in col and 'embeddable' not in col]
TO_TRANSFORM

['tags_joined_emb_0',
 'tags_joined_emb_1',
 'tags_joined_emb_2',
 'tags_joined_emb_3',
 'tags_joined_emb_4',
 'tags_joined_emb_5',
 'tags_joined_emb_6',
 'tags_joined_emb_7',
 'tags_joined_emb_8',
 'tags_joined_emb_9',
 'tags_joined_emb_10',
 'tags_joined_emb_11',
 'tags_joined_emb_12',
 'tags_joined_emb_13',
 'tags_joined_emb_14',
 'tags_joined_emb_15',
 'tags_joined_emb_16',
 'tags_joined_emb_17',
 'tags_joined_emb_18',
 'tags_joined_emb_19',
 'tags_joined_emb_20',
 'tags_joined_emb_21',
 'tags_joined_emb_22',
 'tags_joined_emb_23',
 'tags_joined_emb_24',
 'tags_joined_emb_25',
 'tags_joined_emb_26',
 'tags_joined_emb_27',
 'tags_joined_emb_28',
 'tags_joined_emb_29',
 'tags_joined_emb_30',
 'tags_joined_emb_31',
 'tags_joined_emb_32',
 'tags_joined_emb_33',
 'tags_joined_emb_34',
 'tags_joined_emb_35',
 'tags_joined_emb_36',
 'tags_joined_emb_37',
 'tags_joined_emb_38',
 'tags_joined_emb_39',
 'tags_joined_emb_40',
 'tags_joined_emb_41',
 'tags_joined_emb_42',
 'tags_joined_emb_43'

In [295]:
pca = PCA(n_components=100)
pca_object = pca.fit(df_train[TO_TRANSFORM])
print(pca.explained_variance_ratio_.sum())  
print(pca.explained_variance_ratio_)

0.5715425003683149
[0.12792302 0.03094467 0.02772433 0.0175328  0.01513043 0.01294399
 0.01206901 0.01111525 0.00992805 0.00931963 0.00862945 0.0076869
 0.00747052 0.00729787 0.00659061 0.00634275 0.00605858 0.00573452
 0.00559422 0.00529804 0.00509621 0.00504663 0.00498442 0.00480073
 0.00476571 0.00459104 0.0044883  0.00445272 0.0042858  0.00418003
 0.00414488 0.00407469 0.00393805 0.00388544 0.00375125 0.00364097
 0.00356738 0.00350476 0.00347929 0.00333735 0.00331829 0.00324878
 0.00323679 0.00321931 0.00320325 0.00312163 0.00307264 0.00303028
 0.00299095 0.00297532 0.00292486 0.00286401 0.0028549  0.00279757
 0.00274465 0.00272414 0.00269454 0.00266482 0.00264709 0.00261886
 0.00259688 0.00256326 0.00255259 0.00251773 0.00250953 0.00246805
 0.00243721 0.00242102 0.00238072 0.00237355 0.00236434 0.00234519
 0.00231174 0.00228475 0.00226742 0.00224864 0.00222174 0.00220676
 0.00219037 0.00217782 0.00217069 0.00214091 0.00213359 0.00210861
 0.00208612 0.00207745 0.00206189 0.00205849

In [296]:
n_components = 100
pca_cols = [f"pca_{i}" for i in range(n_components)]

TO_TRANSFORM_vals_train = pca.transform(df_train[TO_TRANSFORM])
TO_TRANSFORM_vals_val   = pca.transform(df_val[TO_TRANSFORM])
TO_TRANSFORM_vals_test  = pca.transform(df_test[TO_TRANSFORM])

df_train = pd.concat([df_train.drop(columns=TO_TRANSFORM), pd.DataFrame(TO_TRANSFORM_vals_train, columns=pca_cols, index=df_train.index)], axis=1)
df_val   = pd.concat([df_val.drop(columns=TO_TRANSFORM),   pd.DataFrame(TO_TRANSFORM_vals_val,   columns=pca_cols, index=df_val.index)],   axis=1)
df_test  = pd.concat([df_test.drop(columns=TO_TRANSFORM),  pd.DataFrame(TO_TRANSFORM_vals_test,  columns=pca_cols, index=df_test.index)],  axis=1)

print(df_train.shape)  
print(df_val.shape)
print(df_test.shape)

(108375, 126)
(36126, 126)
(36126, 126)


# Feature Selection

In [297]:
from sklearn.feature_selection import RFECV
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [241]:
ones  = df_train[df_train["is_trending"] == 1]
zeros = df_train[df_train["is_trending"] == 0].sample(n=len(ones), random_state=42)

sample = pd.concat([ones, zeros]).sample(frac=1, random_state=42)  

print(sample["is_trending"].value_counts())  
print(sample.shape)

is_trending
0    19321
1    19321
Name: count, dtype: int64
(38642, 126)


In [ ]:
# sample = df_train.sample(n=60000, random_state=42)

dt_cf = DecisionTreeClassifier(random_state=42)
rfecv = RFECV(estimator=dt_cf, step=1, cv=5, scoring='f1', n_jobs=-1)
rfecv.fit(sample.drop(columns=["is_trending"]), sample["is_trending"])

selected_cols = df_train.drop(columns=["is_trending"]).columns[rfecv.support_].tolist()
print(f"Selected {len(selected_cols)} features")

Selected 19 features


In [184]:
import json
with open('selected_cols.json', 'w') as file:
    json.dump(selected_cols, file)

In [185]:
df_train.shape

(108375, 126)

In [180]:
from sklearn.metrics import f1_score, classification_report

In [ ]:
X_train = df_train[selected_cols].values
y_train = df_train["is_trending"].values

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

X_test = df_test[selected_cols].values
y_test = df_test["is_trending"].values
y_pred = model.predict(X_test)
accuracy = f1_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(classification_report(y_test, y_pred))

Accuracy: 0.8698328935795955
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     29819
           1       0.98      0.78      0.87      6307

    accuracy                           0.96     36126
   macro avg       0.97      0.89      0.92     36126
weighted avg       0.96      0.96      0.96     36126



/Users/ziadsamer/Documents/who-will-viral/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Feature Scaling

In [196]:
from sklearn.preprocessing import RobustScaler

In [242]:
selected_cols.append("is_trending")

In [244]:
df_train = df_train[selected_cols]
df_val = df_val[selected_cols]
df_test = df_test[selected_cols]

In [245]:
print(df_train.describe())

# More explicit outlier check
Q1 = df_train.quantile(0.25)
Q3 = df_train.quantile(0.75)
IQR = Q3 - Q1
outlier_cols = ((df_train < (Q1 - 1.5*IQR)) | 
                (df_train > (Q3 + 1.5*IQR))).any()
print(f"Columns with outliers: {outlier_cols.sum()}")

          view_count  like_to_view_ratio  comment_to_view_ratio  \
count  108375.000000       108375.000000          108375.000000   
mean       12.883320            0.669785               0.381002   
std         3.209132            0.180227               0.182620   
min         3.103434            0.000000               0.000000   
25%        11.968615            0.644171               0.308174   
50%        13.279576            0.723281               0.423617   
75%        14.849233            0.775760               0.508814   
max        22.150923            0.979767               0.890822   

           tag_count   title_length  description_length  duration_seconds  \
count  108375.000000  108375.000000       108375.000000      1.083750e+05   
mean        9.627303      61.504803          239.842390      1.126116e+04   
std        11.386819      24.182553          520.503749      7.746601e+05   
min         1.000000       1.000000            0.000000      0.000000e+00   
25%        

In [246]:
ALREADY_PROCESSED = [
    "lang_base",
    "lang_region",
    "like_to_view_ratio",       
    "comment_to_view_ratio",    
    "has_cards",
]

In [247]:
COLS_TO_SCALE = [
    c for c in df_train.columns 
    if c not in ALREADY_PROCESSED + [col for col in df_train.columns if 'pca' in col]
    and c != "is_trending"
]

In [248]:
COLS_TO_SCALE

['view_count',
 'tag_count',
 'title_length',
 'description_length',
 'duration_seconds']

In [249]:
scaler = RobustScaler()
df_train[COLS_TO_SCALE] = scaler.fit_transform(df_train[COLS_TO_SCALE])
df_val[COLS_TO_SCALE]   = scaler.transform(df_val[COLS_TO_SCALE])
df_test[COLS_TO_SCALE]  = scaler.transform(df_test[COLS_TO_SCALE])

In [255]:
X_train = df_train.drop(columns=["is_trending"]).values
y_train = df_train["is_trending"].values

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

X_test = df_test.drop(columns=["is_trending"]).values
y_test = df_test["is_trending"].values
y_pred = model.predict(X_test)
accuracy = f1_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(classification_report(y_test, y_pred))

Accuracy: 0.9455900621118012
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     29819
           1       0.99      0.91      0.95      6307

    accuracy                           0.98     36126
   macro avg       0.99      0.95      0.97     36126
weighted avg       0.98      0.98      0.98     36126

